# G.I KO KO — FREE GPU TEXT-TO-VIDEO ENGINE
Generate a short MP4 with the open-source Wan2.1 T2V-1.3B model on a Google Colab GPU. No fal.ai key and no paid inference API. A free GPU is not guaranteed by Colab. Recommended first test: T4 GPU, 480p, 5 seconds.

## 1. Select a GPU runtime
In Colab choose **Runtime → Change runtime type → T4 GPU** if available, then run the cells from top to bottom. GPU access, session length, and usage limits depend on Colab availability.

In [ ]:
import torch, subprocess, shutil
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU is attached. In Colab select Runtime → Change runtime type → T4 GPU, reconnect, and run again.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
if torch.cuda.get_device_properties(0).total_memory < 10 * 1024**3:
    print('WARNING: Less than 10 GB VRAM detected. Generation may run out of memory.')

In [ ]:
%pip -q install -U git+https://github.com/huggingface/diffusers.git transformers accelerate ftfy imageio imageio-ffmpeg

In [ ]:
import torch, gc
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video
from pathlib import Path
MODEL_ID = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
print('Loading open-source model. The first download is several GB and can take time…')
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder='vae', torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL_ID, vae=vae, torch_dtype=torch.float16)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=3.0)
pipe.enable_model_cpu_offload()
print('Wan2.1 model loaded with CPU offload.')

## 2. Enter your video prompt
Use the **Copy Prompt & Open Colab** button in the Android app, then paste the prompt below. Start with a simple English prompt for the first test. The 1.3B model is best tested at 480p.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
prompt_box = widgets.Textarea(
    value='A cinematic aerial shot of a beautiful rural Myanmar village at sunrise, green rice fields, gentle camera movement, realistic lighting, detailed environment',
    placeholder='Paste the prompt copied from G.I KO KO AI VIDEO STUDIO',
    description='Prompt:',
    layout=widgets.Layout(width='100%', height='140px')
)
duration_box = widgets.Dropdown(options=[('5 seconds (recommended)', 5), ('3 seconds (quick test)', 3)], value=5, description='Length:')
seed_box = widgets.IntText(value=42, description='Seed:')
display(prompt_box, duration_box, seed_box)

In [ ]:
import time, traceback
from IPython.display import display, Video
from google.colab import files
def generate_gikoko_video(_):
    prompt = prompt_box.value.strip()
    if not prompt:
        print('Please paste a prompt first.')
        return
    generate_button.disabled = True
    output_path = Path('/content/GIKOKO_WAN21_FREE_GPU.mp4')
    try:
        print('Generating with Wan2.1 T2V-1.3B on the attached GPU…')
        frames = int(duration_box.value * 16) + 1
        generator = torch.Generator(device='cuda').manual_seed(int(seed_box.value))
        result = pipe(
            prompt=prompt,
            negative_prompt='blurry, low quality, distorted, watermark, text, logo, flicker',
            height=480,
            width=832,
            num_frames=frames,
            num_inference_steps=30,
            guidance_scale=6.0,
            generator=generator
        )
        export_to_video(result.frames[0], str(output_path), fps=16)
        if not output_path.exists() or output_path.stat().st_size < 100_000:
            raise RuntimeError('Output MP4 is missing or unexpectedly small.')
        print('Generation complete:', output_path.name, '|', round(output_path.stat().st_size / 1024**2, 2), 'MB')
        display(Video(str(output_path), embed=True))
        print('Starting MP4 download to your phone…')
        files.download(str(output_path))
    except Exception as exc:
        print('Generation failed:', repr(exc))
        print('If this is CUDA out-of-memory, reconnect with a GPU runtime and retry the 3-second test. Colab GPU memory and time are limited.')
    finally:
        generate_button.disabled = False
generate_button = widgets.Button(description='Generate FREE MP4', button_style='success', icon='video')
generate_button.on_click(generate_gikoko_video)
display(generate_button)

## 3. Import the MP4 into Android
When the download finishes, return to G.I KO KO AI VIDEO STUDIO → **Media → Import media** and select `GIKOKO_WAN21_FREE_GPU.mp4`. Add it to your timeline or use the existing cloud FFmpeg renderer for further editing.

## Notes
- No fal.ai API key, FAL_KEY secret, or paid inference API is used by this notebook.
- Colab Free GPU is subject to availability, quota, and session limits. A run can be interrupted.
- The first run downloads the model and may take a long time. Keep the Colab tab/session active.
- Wan2.1 T2V-1.3B is intended for 480p; this notebook outputs 832×480 landscape video. Crop/resize to vertical in the app's render workflow if needed.
- Model source: https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B-Diffusers
- Official project: https://github.com/Wan-Video/Wan2.1
- Check the model and library licenses before redistribution or commercial use.